# 13 — Writing Information Back: Residual MLP Updates

**Description:** Complete the Transformer MLP with a down-projection and residual connection, then inspect how hidden activations write changes into token representations.
**Level:** Beginner
**Tags:** Language Models, Transformers, MLP, Residual Connections, PyTorch

Notebook 12 ended with hidden activations $h$. Those activations live in the wide MLP space; the Transformer stream lives in $d_{model}$ dimensions. This notebook completes the path:

$$x \rightarrow W_{up} \rightarrow \operatorname{GELU} \rightarrow W_{down} \rightarrow \Delta x \rightarrow x+\Delta x$$

The one new idea is **writing back**: the down-projection turns detected features into a residual update.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Recreate the hidden activations

In [ ]:
tokens = ["athlete", "stadium", "running", "teacher"]
X = np.array([[1.0, 0.8, 0.0, 0.2], [0.0, 0.7, 1.0, 0.0], [0.1, 0.6, 0.0, 1.0], [1.0, 0.0, 0.2, 0.2]])
W_up = np.array([[1.0,0.0,0.8,-0.5,0.2,0.7],[1.0,0.8,0.0,0.5,-0.3,0.1],[0.0,1.0,0.0,0.0,0.9,-0.2],[0.0,0.0,1.0,0.7,0.1,0.6]])
b_up = np.array([-1.0,-0.7,-0.6,-0.3,-0.4,-0.5])

def gelu(x):
    return 0.5*x*(1+np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3)))

H = gelu(X @ W_up + b_up)
print("input: ", X.shape)
print("hidden:", H.shape)

## 2. Down-projection returns to model width

The down-projection maps $d_{mlp}$ hidden activations back to $d_{model}$:

$$\Delta X = HW_{down}+b_{down}$$

Its rows can be read as **write directions**: when hidden neuron $j$ activates, row $j$ contributes a particular direction to the update.

In [ ]:
W_down = np.array([
    [ 0.1,  0.8,  0.0,  0.2],  # neuron 0 writes a sport-like direction
    [ 0.0,  0.3,  0.7,  0.0],
    [ 0.2,  0.0,  0.0,  0.8],
    [-0.1,  0.0,  0.0,  0.5],
    [ 0.0, -0.2,  0.8,  0.0],
    [ 0.4,  0.0,  0.0,  0.4],
])
b_down = np.zeros(4)
delta = H @ W_down + b_down

print("H:      ", H.shape)
print("W_down: ", W_down.shape)
print("delta:  ", delta.shape)
print(delta)

## 3. Trace one neuron's contribution

The complete update is a sum of hidden-neuron contributions:

$$\Delta x_i=\sum_j h_{ij}w^{down}_j+b_{down}$$

In [ ]:
position = tokens.index("athlete")
contributions = H[position, :, None] * W_down
print("Contributions to the athlete update:")
for j, contribution in enumerate(contributions):
    print(f"neuron {j} × {H[position, j]:.3f} -> {contribution}")
print("sum:", contributions.sum(axis=0))
assert np.allclose(contributions.sum(axis=0), delta[position])

A detector and a writer are separate pieces. A column of $W_{up}$ determines what activates a neuron; the corresponding row of $W_{down}$ determines what that activation writes.

## 4. The residual connection preserves and updates

Instead of replacing $X$, the block adds its update:

$$X_{new}=X+\Delta X$$

This requires `delta` to have the same shape as `X`. The residual path preserves the old representation while allowing the MLP to modify it.

In [ ]:
X_new = X + delta
print("X shape:    ", X.shape)
print("delta shape:", delta.shape)
print("new shape:  ", X_new.shape)

for token, before, update, after in zip(tokens, X, delta, X_new):
    print(f"{token:>8}: {before} + {update} = {after}")

### Visualize the updates

We plot two selected model-stream coordinates. Arrows show the direction and size of each MLP write.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for i, token in enumerate(tokens):
    ax.scatter(X[i, 0], X[i, 1], color="#4C78A8")
    ax.scatter(X_new[i, 0], X_new[i, 1], color="#F58518")
    ax.annotate("", xy=X_new[i, :2], xytext=X[i, :2], arrowprops={"arrowstyle": "->", "color": "gray"})
    ax.annotate(token, X_new[i, :2], xytext=(5, 5), textcoords="offset points")
ax.scatter([], [], color="#4C78A8", label="before MLP")
ax.scatter([], [], color="#F58518", label="after residual update")
ax.set(xlabel="model feature 1", ylabel="model feature 2", title="The MLP writes an update into each token representation")
ax.legend()
plt.show()

## 5. Scale the residual update

Residual form separates the existing state from the proposed change. Scaling the final down-projection smoothly controls how much the MLP changes the stream.

In [ ]:
for scale in [0.0, 0.25, 0.5, 1.0]:
    candidate = X + scale * delta
    movement = np.linalg.norm(candidate - X, axis=1)
    print(f"scale={scale:.2f} movement={movement}")

At scale zero, the sublayer is an identity map. This easy identity path is one reason residual networks can stack many layers without forcing every sublayer to reconstruct the entire representation.

## 6. Package the complete MLP block in NumPy

In [ ]:
def transformer_mlp(x, W_up, b_up, W_down, b_down):
    hidden = gelu(x @ W_up + b_up)
    update = hidden @ W_down + b_down
    return x + update, {"hidden": hidden, "update": update}

result, cache = transformer_mlp(X, W_up, b_up, W_down, b_down)
assert np.allclose(result, X_new)
print("result shape:", result.shape)
print("cached shapes:", {name: value.shape for name, value in cache.items()})

## 7. The pre-norm PyTorch version

Modern blocks often normalize before the MLP. LayerNorm and the residual stream stay at `d_model`; only the hidden layer expands.

In [ ]:
import torch
from torch import nn

class ResidualMLP(nn.Module):
    def __init__(self, d_model, d_mlp):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.up = nn.Linear(d_model, d_mlp)
        self.activation = nn.GELU()
        self.down = nn.Linear(d_mlp, d_model)

    def forward(self, x, return_update=False):
        update = self.down(self.activation(self.up(self.norm(x))))
        result = x + update
        return (result, update) if return_update else result

torch.manual_seed(13)
block = ResidualMLP(d_model=4, d_mlp=8)
batch = torch.tensor(X, dtype=torch.float32).unsqueeze(0)
result, update = block(batch, return_update=True)
print("input: ", batch.shape)
print("update:", update.shape)
print("result:", result.shape)

## 8. Position-wise does not mean context-free

The MLP does not mix positions, but its input may already contain context gathered by attention. It can therefore detect and transform contextual patterns at each position.

## 9. Challenges

1. Set one row of $W_{down}$ to zero. Which detector can no longer write anything?
2. Make neuron 0 write a negative sport direction and inspect the result.
3. Add a nonzero `b_down`. Which positions receive that write?
4. Verify that the NumPy function accepts a `(batch, tokens, d_model)` input through broadcasting.
5. Insert this `ResidualMLP` after the attention module from Notebook 11.

## Takeaways

- The down-projection converts wide hidden activations into a model-width update.
- Each hidden neuron pairs a detection direction in $W_{up}$ with a write direction in $W_{down}$.
- The residual connection adds the update instead of replacing the representation.
- Attention moves information between positions; the position-wise MLP transforms contextual information already present at each position.
- Notebook 14 uses this detector–writer view to build a toy associative memory.